# Día 2 — El mapa logístico y la ruta al caos

### Taller: Física no lineal en el aula
**Congreso de Profesores de Física — Educación Secundaria**

---

Ayer vimos que un sistema con ecuaciones sencillas puede volverse impredecible.
Hoy vamos al ejemplo más económico posible: **una sola variable, una sola
ecuación, sin derivadas**.

$$x_{n+1} = r\,x_n\,(1 - x_n)$$

Se lee así: si hoy la población (en fracción del máximo posible) es $x_n$,
mañana será $x_{n+1}$. El factor $r$ es la tasa de reproducción, y el $(1-x_n)$
es el freno: cuanta más población hay, menos lugar queda.

Esto se puede hacer **con una calculadora de bolsillo**, y es exactamente por
donde vamos a empezar.

---
**Recordatorio de cómo usar Colab:** *Copiar en Drive* → *Entorno de ejecución →
Ejecutar todas* → esperar. Las celdas van en orden, de arriba hacia abajo.

---
## 0. Preparación

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Checkbox

plt.rcParams["figure.figsize"] = (7, 4.2)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

def logistica(x, r):
    return r*x*(1 - x)

print("Todo listo ✓")

---
## 1. La calculadora: iterar a mano

**Esta parte se hace SIN la computadora.** Saquen el celular, abran la
calculadora, y hagan lo siguiente:

1. Escriban `0.3` (la población inicial).
2. Multiplíquenlo por $r$, y después por $(1 - x)$.
3. Repitan con el resultado. Y otra vez. Y otra vez.

Háganlo unas 20 veces para cada uno de estos valores de $r$, y anoten qué pasa:

| $r$ | ¿Qué observan? |
|---|---|
| 2.8 | |
| 3.2 | |
| 3.5 | |
| 3.9 | |

> **Nota didáctica:** este momento es importante y conviene no apurarlo. Que la
> transición al caos aparezca **en una calculadora de bolsillo**, sin ninguna
> computadora de por medio, es lo que hace creíble todo el resto. Con estudiantes
> funciona muy bien repartir distintos valores de $r$ a distintos grupos y poner
> los resultados en común.

Cuando hayan terminado a mano, ejecuten la celda de abajo para comparar.

In [ ]:
@interact(r=FloatSlider(min=1.0, max=4.0, step=0.01, value=2.8,
                        description="r", continuous_update=False),
          x0=FloatSlider(min=0.01, max=0.99, step=0.01, value=0.30,
                        description="x₀", continuous_update=False),
          n=IntSlider(min=10, max=100, value=40,
                        description="pasos", continuous_update=False))
def iterar(r, x0, n):
    xs = [x0]
    for _ in range(n):
        xs.append(logistica(xs[-1], r))
    xs = np.array(xs)

    plt.figure()
    plt.plot(xs, "o-", ms=4, lw=1)
    plt.xlabel("n (paso)"); plt.ylabel("xₙ"); plt.ylim(-0.02, 1.02)
    plt.title(f"r = {r}")
    plt.show()

    ult = np.round(xs[-16:], 6)
    distintos = sorted(set(ult))
    if len(distintos) == 1:
        print(f"→ Se estabiliza en un valor fijo: x = {distintos[0]:.6f}")
    elif len(distintos) <= 8:
        print(f"→ Ciclo de período {len(distintos)}: {[round(v,4) for v in distintos]}")
    else:
        print(f"→ No se repite: {len(distintos)} valores distintos en los últimos 16 pasos.")

✏️ **Para probar:** recorran $r$ = 2.8 → 3.2 → 3.5 → 3.55 → 3.9 y miren cómo
cambia el mensaje de abajo del gráfico.

Fíjense en algo: **la ecuación no cambió nunca.** Lo único que se movió es un
número. El período va 1 → 2 → 4 → 8 → ... y en algún momento deja de repetirse.
Eso es una **cascada de duplicación de período**.

---
## 2. El diagrama de telaraña: ver la iteración

Hay una forma gráfica muy linda de ver qué está pasando. Dibujamos dos curvas:

- la parábola $y = r\,x(1-x)$,
- la diagonal $y = x$.

Y ahora iteramos gráficamente: desde $x_n$ subimos vertical hasta la parábola
(eso da $x_{n+1}$), y después nos movemos horizontal hasta la diagonal (eso
convierte el resultado en la nueva entrada). Repetir.

Los **puntos fijos** son donde la parábola cruza la diagonal. Que el sistema se
quede ahí o se escape depende de la **pendiente de la parábola** en el cruce:
si $|f'(x^*)| < 1$ atrae, si $|f'(x^*)| > 1$ repele.

In [ ]:
@interact(r=FloatSlider(min=1.0, max=4.0, step=0.01, value=2.8,
                        description="r", continuous_update=False),
          x0=FloatSlider(min=0.01, max=0.99, step=0.01, value=0.20,
                        description="x₀", continuous_update=False),
          pasos=IntSlider(min=1, max=80, value=30,
                        description="pasos", continuous_update=False))
def telarana(r, x0, pasos):
    x = np.linspace(0, 1, 400)
    fig, ax = plt.subplots(figsize=(5.8, 5.8))
    ax.plot(x, logistica(x, r), lw=2, color="steelblue", label=f"y = {r}·x(1−x)")
    ax.plot(x, x, lw=1.2, color="gray", ls="--", label="y = x")

    xi = x0
    for i in range(pasos):
        yi = logistica(xi, r)
        ax.plot([xi, xi], [xi if i else 0, yi], color="crimson", lw=0.9)
        ax.plot([xi, yi], [yi, yi], color="crimson", lw=0.9)
        xi = yi

    # punto fijo no trivial y su estabilidad
    if r > 1:
        xf = 1 - 1/r
        pend = abs(r*(1 - 2*xf))
        estado = "ATRAE" if pend < 1 else "REPELE"
        ax.plot(xf, xf, "ko", ms=8, zorder=5)
        ax.set_title(f"r = {r}    punto fijo x* = {xf:.4f}"
                     f"    |f'(x*)| = {pend:.3f}  →  {estado}")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
    ax.set_xlabel("xₙ"); ax.set_ylabel("xₙ₊₁")
    ax.legend(loc="upper left", fontsize=8)
    plt.show()

✏️ **Para probar:**

1. **$r = 2.8$** — la telaraña se enrosca hacia el cruce. El punto fijo atrae
   ($|f'| = 0.8 < 1$).
2. **$r = 3.2$** — ahora $|f'| = 1.2 > 1$: el punto fijo repele, y la telaraña
   se estabiliza en un **cuadrado** entre dos valores. Ése es el período 2.
3. **$r = 3.5$** — el cuadrado se convierte en una figura más complicada:
   período 4.
4. **$r = 3.9$** — la telaraña llena todo. No hay ciclo.

> **El momento clave es $r = 3$**, cuando $|f'(x^*)|$ cruza exactamente 1. Ahí el
> punto fijo pierde estabilidad y nace el ciclo de período 2. Es una
> **bifurcación**, y se puede calcular a mano: $f'(x^*) = r(1-2x^*) = 2 - r$, que
> vale $-1$ justo en $r = 3$.

---
## 3. El diagrama de bifurcación

En vez de mirar un $r$ por vez, los miramos **todos juntos**. Para cada $r$:

1. iteramos muchas veces y **tiramos** los primeros pasos (el transitorio),
2. graficamos los valores que quedan.

Si el sistema termina en un punto fijo, queda **un** punto. Si termina en período
2, quedan **dos**. Y así.

El truco de velocidad es el mismo de ayer: iteramos **todos los valores de $r$ a
la vez** como un vector de numpy, en lugar de hacer un bucle por cada uno. Por eso
tarda menos de un segundo.

In [ ]:
def diagrama_bifurcacion(r_min=2.5, r_max=4.0, n_r=1500,
                         n_transitorio=500, n_guardar=250):
    r = np.linspace(r_min, r_max, n_r)
    x = np.full(n_r, 0.3)
    for _ in range(n_transitorio):        # descartamos el transitorio
        x = logistica(x, r)
    R, X = [], []
    for _ in range(n_guardar):            # ahora sí, guardamos
        x = logistica(x, r)
        R.append(r.copy()); X.append(x.copy())
    return np.concatenate(R), np.concatenate(X)

In [ ]:
R, X = diagrama_bifurcacion()

plt.figure(figsize=(10, 6))
plt.plot(R, X, ",k", alpha=0.25)
plt.xlabel("r"); plt.ylabel("valores de x a largo plazo")
plt.title("Diagrama de bifurcación del mapa logístico")
plt.xlim(2.5, 4.0); plt.ylim(0, 1); plt.grid(False)
plt.show()

Vale la pena detenerse en este dibujo. De izquierda a derecha:

- hasta $r = 3$: una sola línea (punto fijo);
- en $r = 3$: se parte en dos;
- en $r \approx 3.449$: en cuatro; después en ocho, dieciséis...;
- las bifurcaciones se acumulan cada vez más rápido y en $r \approx 3.5699$ ya
  hay caos;
- adentro del caos hay **ventanas blancas** de orden — la más ancha, cerca de
  $r = 3.83$, es de período 3.

### 3.1 Zoom: la autosemejanza

Hagan zoom sobre una rama cualquiera y comparen con el dibujo completo.

In [ ]:
# ←←← CAMBIAR estos cuatro números para explorar
r_desde,  r_hasta  = 3.54, 3.58
x_desde,  x_hasta  = 0.45, 0.60

R, X = diagrama_bifurcacion(r_min=r_desde, r_max=r_hasta, n_r=1500,
                            n_transitorio=2000, n_guardar=400)
plt.figure(figsize=(9, 5.5))
plt.plot(R, X, ",k", alpha=0.3)
plt.xlim(r_desde, r_hasta); plt.ylim(x_desde, x_hasta)
plt.xlabel("r"); plt.ylabel("x"); plt.grid(False)
plt.title(f"Zoom  r ∈ [{r_desde}, {r_hasta}]")
plt.show()

✏️ **Para probar:** otra región interesante es la ventana de período 3:
`r_desde, r_hasta = 3.847, 3.857` con `x_desde, x_hasta = 0.13, 0.20`. Van a ver
que adentro de la ventana **hay otro diagrama de bifurcación completo**, con su
propia cascada de duplicaciones.

Esto es **autosemejanza**: la misma estructura reaparece a todas las escalas, igual
que en las cuencas del péndulo magnético de ayer. Y no es una coincidencia
estética — es la razón de que exista el número de la próxima sección.

---
## 4. La constante de Feigenbaum

Mirando el diagrama, las bifurcaciones se van juntando. Feigenbaum se preguntó
**a qué ritmo**, y descubrió algo asombroso: el cociente entre intervalos
sucesivos tiende a un número fijo,

$$\delta = \lim_{n\to\infty}\frac{r_n - r_{n-1}}{r_{n+1} - r_n} = 4.669201609\ldots$$

y ese número **no depende del mapa**. Sale igual para $x(1-x)$, para $\sin(\pi x)$,
y para el goteo de una canilla o un circuito electrónico. Es una constante
universal, como $\pi$, pero de las bifurcaciones.

### Cómo lo calculamos

Localizar los $r_n$ exactos (donde el ciclo cambia de estabilidad) es numéricamente
incómodo. Hay un truco mucho más limpio: en vez de los puntos de bifurcación,
usamos los **ciclos superestables** $R_n$ — los valores de $r$ donde el ciclo de
período $2^n$ pasa justo por $x = 0.5$, es decir

$$f^{(2^n)}(0.5) = 0.5$$

Eso es una ecuación de una variable que se resuelve por **bisección**, el método
más elemental que hay. Los $R_n$ convergen al mismo $\delta$ y son mucho más
fáciles de encontrar con precisión.

In [ ]:
def iterar_veces(x, r, veces):
    for _ in range(veces):
        x = logistica(x, r)
    return x

def buscar_superestable(n, r_lo, r_hi, n_muestras=4000):
    "Encuentra R_n resolviendo f^(2^n)(0.5) = 0.5 por bisección."
    m = 2**n
    g = lambda r: iterar_veces(0.5, r, m) - 0.5

    # 1) rastrillamos el intervalo buscando un cambio de signo
    rs = np.linspace(r_lo, r_hi, n_muestras)
    gs = np.array([g(r) for r in rs])
    cruces = np.where(np.sign(gs[:-1])*np.sign(gs[1:]) < 0)[0]
    if len(cruces) == 0:
        return None
    a, b = rs[cruces[0]], rs[cruces[0] + 1]

    # 2) bisección clásica
    ga = g(a)
    for _ in range(200):
        c = 0.5*(a + b)
        gc = g(c)
        if ga*gc <= 0:
            b = c
        else:
            a, ga = c, gc
        if b - a < 1e-15:
            break
    return 0.5*(a + b)

In [ ]:
n_max = 9      # ←←← hasta período 2^9 = 512

Rs = [2.0]     # R_1 = 2 exacto: ahí el ciclo de período 2 es superestable
print(f"  n=1   período   2    R_n = {Rs[0]:.13f}")

for n in range(2, n_max + 1):
    if n == 2:
        lo, hi = 3.1, 3.3
    else:
        # usamos la propia escala de Feigenbaum para adivinar dónde buscar
        paso = Rs[-1] - Rs[-2]
        lo   = Rs[-1] + 1e-9
        hi   = Rs[-1] + 2.2*paso/4.669201609
    R = buscar_superestable(n, lo, hi)
    if R is None:
        print(f"  n={n}: no se encontró (precisión de float agotada)")
        break
    Rs.append(R)
    print(f"  n={n}   período {2**n:3d}    R_n = {R:.13f}")

print("\n  δ_n = (R_n − R_{n−1}) / (R_{n+1} − R_n)\n")
for i in range(1, len(Rs) - 1):
    d = (Rs[i] - Rs[i-1])/(Rs[i+1] - Rs[i])
    print(f"    n={i+1}:   δ = {d:.9f}     error = {abs(d - 4.669201609102990):.1e}")

print(f"\n  valor exacto:  δ = 4.669201609102990...")

Los valores convergen: 4.7089 → 4.6808 → 4.6630 → 4.6684 → 4.66895 → 4.669157 →
**4.669191**, con un error de $10^{-5}$ contra el valor exacto.

Y hay un límite real acá que conviene mostrar en vez de esconder: pasando de
$n = 9$ la búsqueda empieza a fallar. No es un error del programa — es que iterar
$2^{10} = 1024$ veces amplifica el error de redondeo de la aritmética de la
computadora hasta tapar las diferencias, que ya son del orden de $10^{-6}$. Para
seguir haría falta aritmética de precisión extendida.

> **Buen tema de discusión:** el sistema que estamos estudiando amplifica
> diferencias pequeñas. La computadora que lo estudia también tiene diferencias
> pequeñas (el redondeo). En algún punto se encuentran.

🧩 **Ejercicio 1 — la universalidad**

Feigenbaum dice que $\delta$ **no depende del mapa**, siempre que tenga un máximo
suave. Vamos a comprobarlo con un mapa completamente distinto:

$$x_{n+1} = r\,\sin(\pi x_n)$$

No tiene nada que ver con una parábola. En la celda de abajo está todo hecho:
sólo hay que **descomentar la línea del mapa seno** (borrar el `#`) y ejecutar.

In [ ]:
# --- elegir el mapa: dejar UNA sola línea sin comentar ---
mapa = lambda x, r: r*x*(1-x)          # logístico
# mapa = lambda x, r: r*np.sin(np.pi*x)  # ←←← DESCOMENTAR ESTA

# ---- de acá para abajo no hay que tocar nada ----
def super_generico(n, r_lo, r_hi, f, n_muestras=4000):
    m = 2**n
    def g(r):
        x = 0.5
        for _ in range(m):
            x = f(x, r)
        return x - 0.5
    rs = np.linspace(r_lo, r_hi, n_muestras)
    gs = np.array([g(r) for r in rs])
    c = np.where(np.sign(gs[:-1])*np.sign(gs[1:]) < 0)[0]
    if len(c) == 0:
        return None
    a, b = rs[c[0]], rs[c[0]+1]
    ga = g(a)
    for _ in range(200):
        mm = 0.5*(a+b); gm = g(mm)
        if ga*gm <= 0: b = mm
        else: a, ga = mm, gm
        if b - a < 1e-15: break
    return 0.5*(a+b)

# El rango de r es distinto en cada mapa, asi que arrancamos con una
# busqueda amplia y despues nos vamos guiando por la propia escala 4.669.
r_ini, r_fin = (2.5, 4.0) if mapa(0.5, 1.0) < 0.9 else (0.5, 1.0)

Rg = []
lo, hi = r_ini, r_fin
for n in range(1, 9):
    R = super_generico(n, lo, hi, mapa)
    if R is None:
        break
    Rg.append(R)
    lo = R + 1e-9
    hi = R + (0.35*(r_fin - r_ini) if len(Rg) < 2
              else 2.2*(Rg[-1] - Rg[-2])/4.6692)

for i, R in enumerate(Rg, start=1):
    print(f"  n={i}  período {2**i:4d}   R_n = {R:.12f}")
print()
for i in range(1, len(Rg)-1):
    print(f"    δ = {(Rg[i]-Rg[i-1])/(Rg[i+1]-Rg[i]):.6f}")

Los $R_n$ del mapa seno son **números totalmente distintos** a los del logístico
— pero los $\delta$ convergen al **mismo 4.669**.

Ese es el resultado profundo del día: la ruta al caos por duplicación de período
tiene una estructura cuantitativa que **no depende de los detalles del sistema**.
Por eso el mismo número aparece midiendo convección en helio líquido, un circuito
con un diodo, o una canilla que gotea. Y por eso tiene sentido enseñarlo con una
parábola.

🧩 **Ejercicio 2 — encontrar la ventana de período 3**

En el diagrama de bifurcación, adentro de la zona caótica, hay una franja blanca
ancha. Encuéntrenla ajustando el valor de abajo hasta que el programa diga
**período 3**.

*(Pista: está entre 3.82 y 3.86.)*

In [ ]:
r_ventana = 3.80      # ←←← CAMBIAR hasta encontrar el período 3

x = 0.3
for _ in range(5000):
    x = logistica(x, r_ventana)
orb = [x := logistica(x, r_ventana) for _ in range(60)]
distintos = sorted(set(np.round(orb, 6)))

if len(distintos) <= 12:
    print(f"r = {r_ventana}  →  PERÍODO {len(distintos)}")
    print(f"   valores: {[round(v, 5) for v in distintos]}")
else:
    print(f"r = {r_ventana}  →  caótico (sin período detectable)")

> **El teorema de Sharkovskii** dice algo notable sobre esto: si un mapa continuo
> tiene una órbita de período 3, entonces tiene órbitas de **todos** los períodos.
> De ahí el título del célebre trabajo de Li y Yorke de 1975, *"Period three
> implies chaos"* — el artículo que además le puso ese nombre al campo.

---
## 5. Para llevarse

| Concepto | Dónde apareció |
|---|---|
| Iteración y punto fijo | la calculadora, la telaraña |
| Estabilidad = pendiente | $|f'(x^*)| < 1$ atrae, $> 1$ repele |
| Bifurcación | en $r = 3$, cuando $f'(x^*) = -1$ |
| Cascada de duplicaciones | el diagrama completo |
| Autosemejanza | los zooms |
| Universalidad | $\delta = 4.669$ igual para el mapa seno |

Dos ideas para cerrar:

1. **La complejidad no requiere ecuaciones complicadas.** Todo lo de hoy salió de
   $x(1-x)$, que es lo más simple que hay después de una recta. Un estudiante de
   secundaria puede reproducir la transición al caos con una calculadora.
2. **Hay orden adentro del caos, y es medible.** $\delta = 4.669$ es una
   predicción cuantitativa, verificable en el laboratorio, sobre sistemas que a
   primera vista no tienen nada en común.

**Mañana:** el sistema de Lorenz. Volvemos a ecuaciones diferenciales, ahora en
tres dimensiones, para ver el atractor extraño y medir el exponente de Lyapunov
— que es la versión continua de lo que hoy vimos como "sensibilidad".